# 01 EDA (member A)

Goal: understand the data and find the problems to fix. Every figure is saved to `reports/figures/` for the report and slides.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
from src.data import load_data
train, test = load_data()
print(train.shape, test.shape)
train.head()

## 1. Target: SalePrice
Right-skewed -> close to normal after log (iter1 uses log; control: iter1_no_log).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], kde=True, ax=ax[0]).set_title(f"SalePrice (skew={train[TARGET].skew():.2f})")
sns.histplot(np.log1p(train[TARGET]), kde=True, ax=ax[1]).set_title(f"log1p(SalePrice) (skew={np.log1p(train[TARGET]).skew():.2f})")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "target_distribution.png", dpi=150)

## 2. Features most correlated with price

In [ ]:
corr = train.select_dtypes("number").corr()[TARGET].drop(TARGET).sort_values(ascending=False)
top = corr.head(10)
print(top)
plt.figure(figsize=(8, 4)); sns.barplot(x=top.values, y=top.index)
plt.title("Top 10 correlation with SalePrice"); plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_correlations.png", dpi=150)

In [ ]:
cols = [TARGET] + list(top.index[:6])
plt.figure(figsize=(9, 7)); sns.heatmap(train[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "corr_heatmap.png", dpi=150)

## 3. Outliers
Bottom right: huge living area but low price (removed in iter2).

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=train, x="GrLivArea", y=TARGET)
out = train[(train.GrLivArea > 4000) & (train[TARGET] < 300000)]
plt.scatter(out.GrLivArea, out[TARGET], color="red", s=80, label="outliers"); plt.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR / "outliers.png", dpi=150)
out[["Id", "GrLivArea", TARGET]]

## 4. Missing values
Note: many NAs mean "no such facility", see `data_description.txt` (handled by fill_missing in iter2).

In [ ]:
full = pd.concat([train.drop(columns=TARGET), test])
missing = (full.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
plt.figure(figsize=(8, 8)); sns.barplot(x=missing.values, y=missing.index)
plt.xlabel("% missing"); plt.tight_layout(); plt.savefig(FIGURE_DIR / "missing_values.png", dpi=150)
missing

## 5. Categorical features: e.g. Neighborhood / OverallQual

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
order = train.groupby("Neighborhood")[TARGET].median().sort_values().index
sns.boxplot(data=train, x="Neighborhood", y=TARGET, order=order, ax=ax[0]); ax[0].tick_params(axis="x", rotation=90)
sns.boxplot(data=train, x="OverallQual", y=TARGET, ax=ax[1])
plt.tight_layout(); plt.savefig(FIGURE_DIR / "categorical_vs_price.png", dpi=150)

## Findings (for report section 2)

- 
- 
- 